In [60]:
import pandas as pd
from collections import deque

# =========================
# LOAD DATA
# =========================
matches = pd.read_csv("../data/processed/matches_cleaned.csv")

# Ensure correct order
matches['date'] = pd.to_datetime(matches['date'])
matches = matches.sort_values('date').reset_index(drop=True)

# =========================
# INITIALIZE FEATURES
# =========================
matches['team1_win_rate'] = 0.0
matches['team2_win_rate'] = 0.0
matches['head_to_head'] = 0
matches['team1_recent_form'] = 0
matches['team2_recent_form'] = 0

# 🔥 NEW
matches['team1_matches_played'] = 0
matches['team2_matches_played'] = 0

team_stats = {}
head2head = {}
recent_form = {}

# =========================
# FEATURE ENGINEERING LOOP
# =========================
for i in range(len(matches)):
    team1 = matches.loc[i, 'team1']
    team2 = matches.loc[i, 'team2']
    winner = matches.loc[i, 'winner']

    if team1 not in team_stats:
        team_stats[team1] = [0, 0]
    if team2 not in team_stats:
        team_stats[team2] = [0, 0]

    t1_wins, t1_matches = team_stats[team1]
    t2_wins, t2_matches = team_stats[team2]

    # Win rate
    matches.loc[i, 'team1_win_rate'] = t1_wins / t1_matches if t1_matches > 0 else 0.5
    matches.loc[i, 'team2_win_rate'] = t2_wins / t2_matches if t2_matches > 0 else 0.5

    # 🔥 EXPERIENCE FEATURE
    matches.loc[i, 'team1_matches_played'] = t1_matches
    matches.loc[i, 'team2_matches_played'] = t2_matches

    # Head-to-head
    pair = tuple(sorted([team1, team2]))
    if pair not in head2head:
        head2head[pair] = {team1: 0, team2: 0}

    matches.loc[i, 'head_to_head'] = (
        head2head[pair].get(team1, 0) - head2head[pair].get(team2, 0)
    )

    # Recent form
    if team1 not in recent_form:
        recent_form[team1] = deque(maxlen=5)
    if team2 not in recent_form:
        recent_form[team2] = deque(maxlen=5)

    matches.loc[i, 'team1_recent_form'] = sum(recent_form[team1])
    matches.loc[i, 'team2_recent_form'] = sum(recent_form[team2])

    # =========================
    # UPDATE AFTER FEATURE CREATION
    # =========================
    team_stats[team1][1] += 1
    team_stats[team2][1] += 1

    if winner == team1:
        team_stats[team1][0] += 1
    elif winner == team2:
        team_stats[team2][0] += 1

    if winner in head2head[pair]:
        head2head[pair][winner] += 1

    recent_form[team1].append(1 if winner == team1 else 0)
    recent_form[team2].append(1 if winner == team2 else 0)

# =========================
# ADDITIONAL FEATURES
# =========================
matches['win_rate_diff'] = matches['team1_win_rate'] - matches['team2_win_rate']
matches['recent_form_diff'] = matches['team1_recent_form'] - matches['team2_recent_form']
matches['h2h_diff'] = matches['head_to_head'] / 10.0

# 🔥 NEW
matches['experience_diff'] = matches['team1_matches_played'] - matches['team2_matches_played']

# Target
matches['team1_win'] = (matches['winner'] == matches['team1']).astype(int)

# Toss feature
matches['toss_winner_is_team1'] = (matches['toss_winner'] == matches['team1']).astype(int)

# =========================
# ENCODING
# =========================
matches['venue'] = matches['venue'].astype('category').cat.codes
matches['toss_decision'] = matches['toss_decision'].map({'bat': 1, 'field': 0})

# =========================
# REMOVE INITIAL NOISE
# =========================
matches = matches.iloc[200:].reset_index(drop=True)

# =========================
# FINAL FEATURES
# =========================
features = matches[[
    'toss_winner_is_team1',
    'toss_decision',
    'venue',
    'team1_win_rate',
    'team2_win_rate',
    'win_rate_diff',
    'recent_form_diff',
    'h2h_diff',
    'experience_diff'
]]

target = matches['team1_win']

# =========================
# SAVE
# =========================
features.to_csv("../data/feature_store/match_features_v2.csv", index=False)
target.to_csv("../data/feature_store/match_target_v2.csv", index=False)

# =========================
# DEBUG
# =========================
print("✅ Features saved successfully")
print("Shape:", features.shape)
print(features.head())

✅ Features saved successfully
Shape: (691, 9)
   toss_winner_is_team1  toss_decision  venue  team1_win_rate  team2_win_rate  \
0                     1              0     51        0.568966        0.418182   
1                     1              0     17        0.464286        0.583333   
2                     1              0     16        0.518519        0.466667   
3                     0              0     17        0.473684        0.509091   
4                     1              1     43        0.473684        0.559322   

   win_rate_diff  recent_form_diff  h2h_diff  experience_diff  
0       0.150784                 0       0.7                3  
1      -0.119048                -1      -0.3               -4  
2       0.051852                 1       0.2               -6  
3      -0.035407                 1       0.1                2  
4      -0.085638                -2       0.0               -2  
